# Лабораторная работа №6. IPv4: адрес, сеть и префикс

В Lab 005 Linux узнал MAC локального соседа с помощью ARP. Но **как хост вообще решает, локальный ли destination IP и понадобится ли ARP на этом интерфейсе?** Это решают префикс интерфейса и таблица маршрутов, прежде чем Ethernet-кадр будет сформирован.

**Маршрут лабораторной:** вычислить границы /24 и /25 → проверить реальный connected route → захватить работающий ICMP/ARP → изменить префикс **только pc1** → увидеть одностороннее представление о «локальности» → диагностировать и восстановить → самостоятельно спроектировать VLSM.

## Один Ethernet-линк, две независимые routing table

![Как /24 и /25 меняют выбор исходящего интерфейса](assets/prefix.svg)

`pc1:eth1 ↔ pc2:eth1` — прямое L2-соединение. Не требуется switch или router. При `/24` оба считают адрес друг друга on-link. Во время поломки **только pc1** получает `192.168.50.10/25`, а `pc2` остаётся с `192.168.50.130/24`. Поэтому правила отправки у них становятся **асимметричными**.

Все действия ниже относятся к учебному `eth1`. `eth0` — management plane платформы, его адреса и маршруты **не удаляем и не меняем**.

## 1. Сначала прогноз — без команд на хостах

Сеть определяется побитовой операцией `IP AND mask`. `/24` оставляет под host-часть 8 бит, `/25` — 7 бит. Сравните `192.168.50.10` и `192.168.50.130`: оба имеют общий /24, но старший бит последнего октета у них различен.

Проверьте расчёт с помощью стандартного Python `ipaddress` — затем сравним его с **Linux FIB**, а не заменим им сетевой эксперимент.

In [ ]:
import ipaddress
from IPython.display import display

pc1 = ipaddress.ip_address("192.168.50.10")
pc2 = ipaddress.ip_address("192.168.50.130")

for length in (23, 24, 25, 26):
    network = ipaddress.ip_network(f"{pc1}/{length}", strict=False)
    print(f"/{length}: network={str(network):22} broadcast={str(network.broadcast_address):17} "
          f"pc2_is_on_link={pc2 in network}")


### Q1. Для какого префикса из `/23`, `/24`, `/25`, `/26` адрес `192.168.50.130` перестанет быть локальным относительно `192.168.50.10`? Обоснуйте через границу сети, а не только Python-вывод.

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверить рассуждение (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q1 непосредственно выше. Оцени корректность CIDR, longest-prefix match и роли IPv4 маршрута, отметь упущенный шаг и задай 1–2 уточняющих вопроса. **Не раскрывай готовый ответ, не переписывай его и не выдавай готовые команды для следующих этапов.**

</details>

## 2. Настраиваем /24 и сравниваем расчёт с Linux

В terminal **pc1**:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 192.168.50.10/24 dev eth1
ip -4 -o addr show dev eth1
ip -4 route show dev eth1
ip -4 route get 192.168.50.130 from 192.168.50.10
```

В terminal **pc2**:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 192.168.50.130/24 dev eth1
ip -4 -o addr show dev eth1
ip -4 route show dev eth1
ip -4 route get 192.168.50.10 from 192.168.50.130
```

В обеих таблицах появится connected network `192.168.50.0/24` с `dev eth1`. `ip route get` для другой стороны должен выбирать **eth1 без `via`**. `via` обозначал бы next-hop router; здесь он не нужен.

### Q2. Какая запись connected route появилась после установки `192.168.50.10/24` и как вывод `ip route get` показывает, что пакет к `.130` отправляется без gateway?

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверить рассуждение (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q2 непосредственно выше. Оцени корректность CIDR, longest-prefix match и роли IPv4 маршрута, отметь упущенный шаг и задай 1–2 уточняющих вопроса. **Не раскрывай готовый ответ, не переписывай его и не выдавай готовые команды для следующих этапов.**

</details>

### Первый настоящий захват: /24, успешная доставка

На `pc1` перед захватом удалите только neighbor-запись второй стороны:

```bash
ip neigh del 192.168.50.130 dev eth1 2>/dev/null || true
ip neigh show 192.168.50.130 dev eth1
```

Запустите следующую ячейку, затем **пока она работает** выполните в terminal `pc1`:

```bash
ping -c 3 -W 1 192.168.50.130
```

Мы специально слушаем `pc1:eth1`, чтобы увидеть ARP и ICMP именно на **учебном линке**, а не на management-интерфейсе.

In [ ]:
%%capture_traffic pc1:eth1 --filter "arp or icmp" --timeout 12 --packets 80 --save captures/prefix-24-working.pcap
import time
print("Пока идёт capture, на pc1: ping -c 3 -W 1 192.168.50.130")
time.sleep(10)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

good = PcapViewer("captures/prefix-24-working.pcap", "arp or icmp", limit=60)
good_frames = good.packets()
display(good_frames)
if good_frames.empty:
    print("PCAP пуст: запустите ping одновременно с capture.")
else:
    print("Первые реальные протоколы /24:")
    print("\n".join(good.packet(int(good_frames.iloc[0]["№"]), "protocols").splitlines()[:55]))


Свяжите **три свидетельства**: `ip route get` → ARP на `pc1:eth1` → успешный ICMP Echo Reply. Конкретное количество ARP не фиксировано: если у другой стороны сохранилась neighbor-запись, она может обойтись без нового запроса; подробнее это было в Lab 005.

## 3. Предсказываем отказ: /24 → /25 только у pc1

**Не меняйте pc2.** Сначала предскажите, что теперь скажет `pc1` о маршруте к `192.168.50.130`, а что скажет `pc2` о маршруте к `192.168.50.10`.

### Q3. После изменения только `pc1: 192.168.50.10/24 → /25` что покажет `ip route get` на каждой стороне? Почему у двух хостов на одном проводе получаются разные решения о локальности?

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверить рассуждение (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q3 непосредственно выше. Оцени корректность CIDR, longest-prefix match и роли IPv4 маршрута, отметь упущенный шаг и задай 1–2 уточняющих вопроса. **Не раскрывай готовый ответ, не переписывай его и не выдавай готовые команды для следующих этапов.**

</details>

На **pc1** поменяйте только IPv4 prefix:

```bash
ip addr del 192.168.50.10/24 dev eth1
ip addr add 192.168.50.10/25 dev eth1
ip -4 route show dev eth1
ip -4 route get 192.168.50.130 from 192.168.50.10
```

Теперь локальная сеть pc1 — `192.168.50.0/25`. Адрес `.130` относится к **другой** /25. Поэтому вывод `ip route get` больше не должен показывать direct on-link route `dev eth1` для этого адреса.

**Важная особенность C9s:** у контейнера может сохраняться служебный default через `eth0`. `ip route get` в таком случае выберет **management fallback** либо вернёт `Network is unreachable`. Это не путь учебной сети и **не повод удалять default route**. В этой лаборатории отрицательный опыт наблюдаем на `eth1` и диагностируем по FIB.

На **pc2** ничего не изменилось; проверьте:

```bash
ip -4 route get 192.168.50.10 from 192.168.50.130
```

Он всё ещё считает `pc1` локальным, поскольку его адрес остался `/24`.

### Что происходит на проводе при несовпадающих префиксах?

Запустите capture на **pc1:eth1**. Затем в terminal **pc2**, пока идёт запись:

```bash
ping -c 4 -W 1 192.168.50.10
```

`pc2` считает `pc1` on-link и может отправить ARP и ICMP на учебный линк. `pc1` при этом **не имеет прямого обратного маршрута к pc2 через eth1**, поэтому успешную двустороннюю доставку нельзя предполагать.

В capture могут быть входящие Echo Request, но не исходящие Echo Reply по `eth1`; поведение зависит также от neighbor-cache и фильтрации ядра (например, reverse-path filtering). **Главное доказательство — вывод двух `ip route get`**, а не требование одинакового количества пакетов у всех участников.

In [ ]:
%%capture_traffic pc1:eth1 --filter "arp or icmp" --timeout 13 --packets 90 --save captures/prefix-25-asymmetric.pcap
import time
print("Пока идёт capture, на pc2: ping -c 4 -W 1 192.168.50.10")
time.sleep(11)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

broken = PcapViewer("captures/prefix-25-asymmetric.pcap", "arp or icmp", limit=60)
broken_frames = broken.packets()
display(broken_frames)
if broken_frames.empty:
    print("Не видно ARP/ICMP на pc1:eth1. Проверьте время capture и отдельно сравните route get на обеих сторонах.")
else:
    for number in broken_frames["№"].head(3):
        print(f"\nFrame {int(number)}: смотрим IP source/destination и направление")
        print("\n".join(broken.packet(int(number), "protocols").splitlines()[:48]))


### Q4. Если `pc2` отправил Echo Request на `pc1:eth1`, почему полученный кадр ещё не означает, что `pc1` сможет отправить Echo Reply обратно через тот же интерфейс? Какие команды или поля PCAP это показывают?

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверить рассуждение (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q4 непосредственно выше. Оцени корректность CIDR, longest-prefix match и роли IPv4 маршрута, отметь упущенный шаг и задай 1–2 уточняющих вопроса. **Не раскрывай готовый ответ, не переписывай его и не выдавай готовые команды для следующих этапов.**

</details>

## 4. Исправляем только изменённый prefix

На **pc1**:

```bash
ip addr del 192.168.50.10/25 dev eth1
ip addr add 192.168.50.10/24 dev eth1
ip -4 route show dev eth1
ip -4 route get 192.168.50.130 from 192.168.50.10
ping -c 3 -W 1 192.168.50.130
```

На **pc2**:

```bash
ping -c 2 -W 1 192.168.50.10
```

Теперь у обоих есть прямой connected route к другой стороне на `eth1`. Проверяем оба направления, чтобы не принять **односторонний путь** за исправленную сеть.

## 5. CHECK · маршрут и доставка

Запустите CHECK после восстановления /24. Он проверяет не историю команд, а наблюдаемое состояние: **оба IPv4 /24**, их connected route, direct route lookup без `via` и **ICMP в обе стороны**. Ничего в `eth0` менять не требуется.

Если CHECK красный, сравните сначала `ip -4 -o addr show dev eth1`, затем `ip -4 route show dev eth1`, `ip -4 route get`, только после этого `ping`. Не пытайтесь чинить адреса путём изменения management default.

## 6. VLSM: несколько сетей вместо одной большой

Теперь отдельная задача на бумаге и в Python, **без изменения работающей топологии**. Есть диапазон `192.168.60.0/24`; нужно выделить непересекающиеся сети:

- сегмент A — **70** устройств;
- сегмент B — **35** устройств;
- сегмент C — **14** устройств.

Для каждого сегмента выберите **минимальный подходящий IPv4 prefix**, учитывая network и broadcast address. Размещайте сети от самой большой к меньшей, на корректных границах. Не путайте `2^(32-prefix)` адресов всего и пригодные для обычных хостов адреса `2^(32-prefix) - 2`.

До запуска проверочного Python запишите собственное предложение (prefix, network, usable host range, broadcast).

### Q5. Как распределить внутри `192.168.60.0/24` подсети для 70, 35 и 14 обычных IPv4-хостов без пересечений? Укажите для каждой network/prefix и объясните, почему выбранный prefix минимально достаточен.

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверить рассуждение (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q5 непосредственно выше. Оцени корректность CIDR, longest-prefix match и роли IPv4 маршрута, отметь упущенный шаг и задай 1–2 уточняющих вопроса. **Не раскрывай готовый ответ, не переписывай его и не выдавай готовые команды для следующих этапов.**

</details>

### Проверка дизайна на схеме

![Пример непересекающегося разбиения VLSM](assets/vlsm.svg)

Схема показывает **один корректный вариант размещения**. Сравните с собственной конструкцией: другие границы допустимы, если сеть начинается на корректной границе prefix и диапазоны не пересекаются.

In [ ]:
import ipaddress

# Эталон одного корректного распределения — сравните после своего ответа.
segments = [
    ("A, 70 hosts", "192.168.60.0/25", 70),
    ("B, 35 hosts", "192.168.60.128/26", 35),
    ("C, 14 hosts", "192.168.60.192/28", 14),
]
parent = ipaddress.ip_network("192.168.60.0/24")
networks = []
for name, cidr, needed in segments:
    net = ipaddress.ip_network(cidr)
    available = net.num_addresses - 2
    assert net.subnet_of(parent), f"{net} выходит за /24"
    assert available >= needed, f"{net} недостаточна для {needed} хостов"
    assert all(not net.overlaps(existing) for existing in networks), f"{net} пересекается"
    networks.append(net)
    print(f"{name:12} {str(net):20} available={available:3} "
          f"first={net.network_address+1} last={net.broadcast_address-1} "
          f"broadcast={net.broadcast_address}")
print(f"Total addresses allocated: {sum(n.num_addresses for n in networks)} / {parent.num_addresses}")


### Q6. Почему при VLSM нельзя просто записать три произвольных IP-диапазона подряд? Объясните требования к границам CIDR, непересечению сетей и отличию объёма всего блока от числа используемых хостовых адресов.

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверить рассуждение (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q6 непосредственно выше. Оцени корректность CIDR, longest-prefix match и роли IPv4 маршрута, отметь упущенный шаг и задай 1–2 уточняющих вопроса. **Не раскрывай готовый ответ, не переписывай его и не выдавай готовые команды для следующих этапов.**

</details>

## Итог Lab 006

- `IP AND mask` определяет адрес сети; prefix length определяет, какие IPv4 являются on-link.
- Добавление `192.168.50.10/24` создаёт connected route для `192.168.50.0/24`.
- `/24 → /25` на **одном** хосте меняет его локальную сеть, хотя физический Ethernet остаётся тем же.
- Решения о маршруте независимы на `pc1` и `pc2`; входящий Echo Request не гарантирует обратный путь.
- `ip route get` и PCAP вместе показывают различие L3 routing и L2 reachability.
- Management default на `eth0` — инфраструктурный маршрут, не «правильный gateway» учебного линка.
- VLSM позволяет выделять подсети по числу хостов; каждая подсеть обязана иметь корректную границу и не пересекаться с другими.

Вы восстановили /24, проверили связность в оба направления и объяснили, почему совпадение **L2-сегмента** ещё не означает совпадение **IPv4-сети**.